In [ ]:
# Working script!

"""
freezeup_manual_review.py
─────────────────────────────
Interactive OpenCV tool for manually reviewing and correcting freeze-up dates.

CONTROLS
  ←  / A      Move freeze-up divider left  (one image)
  →  / D      Move freeze-up divider right (one image)
  Enter        Confirm position and save result, advance to next lake-year
  X            Mark as NO MANUAL DATE (no detectable freeze-up), advance
  B            Go back one lake-year (undo last save)
  O            Toggle lake outline overlay
  Z            Toggle tight zoom (40 m buffer vs 500 m default)
  ESC          Quit (progress already saved up to this point)

OUTPUT CSV columns
  study_site | lake_id | year | original_freezeup_date |
  manual_freezeup_date | image_before | image_after
"""

import os
import glob
import queue
import threading
import warnings
import csv
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import geopandas as gpd
import rasterio
from rasterio.mask import mask as rio_mask

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
#  USER CONFIG
#  1. Run  `ls /Volumes/`  in Terminal to find your drive name.
#  2. Replace YOUR_DRIVE_NAME below with that name (exact capitalisation).
#  3. Verify the YF / YKD shapefile folder names match what is on disk.
# ══════════════════════════════════════════════════════════════════════════════

_ROOT = "/Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series"
_CSVS = os.path.join(_ROOT, "Freezeup CSVs")
_IMGS = os.path.join(_ROOT, "Imagery")
_SHP  = os.path.join(_ROOT, "Shapefiles")

TIMESERIES_CSVS = {
    "NS_50x50":  os.path.join(_CSVS, "NS_freezeup_timeseries.csv"),
    "YF_50x50":  os.path.join(_CSVS, "YF_freezeup_timeseries.csv"),
    "YKD_50x50": os.path.join(_CSVS, "YKD_freezeup_timeseries.csv"),
}

FREEZEUP_RESULTS_CSVS = {
    "NS_50x50":  os.path.join(_CSVS, "NS_freezeup_results.csv"),
    "YF_50x50":  os.path.join(_CSVS, "YF_freezeup_results.csv"),
    "YKD_50x50": os.path.join(_CSVS, "YKD_freezeup_results.csv"),
}

CLOUD_CSVS = [
    os.path.join(_CSVS, "NS_cloud_classifications.csv"),
    os.path.join(_CSVS, "YF_cloud_classifications.csv"),
    os.path.join(_CSVS, "YKD_cloud_classifications.csv"),
]

SITE_FOLDERS = {
    "NS_50x50":  os.path.join(_IMGS, "NS 50x50 km"),
    "YF_50x50":  os.path.join(_IMGS, "YF 50x50 km"),
    "YKD_50x50": os.path.join(_IMGS, "YKD 50x50 km"),
}

SITE_SHAPEFILES = {
    "NS_50x50":  os.path.join(_SHP, "NS Lakes from ALPOD Shapefile", "NS_50x50km_lakes.shp"),
    "YF_50x50":  os.path.join(_SHP, "YF Lakes from ALPOD Shapefile", "YF_50x50km_lakes.shp"),
    "YKD_50x50": os.path.join(_SHP, "YKD Lakes from ALPOD Shapefile", "YKD_50x50km_lakes.shp"),
}

OUTPUT_CSV = os.path.join(_CSVS, "manual_freezeup_review.csv")

# ── Display ──────────────────────────────────────────────────────────────────
MAX_WINDOW_W   = 1800
MAX_WINDOW_H   = 900
DIVIDER_W      = 44
LABEL_H        = 62
HEADER_H       = 52
MIN_THUMB_W    = 90

FREEZE_THRESHOLD   = 75.0
BOUNDARY_BUFFER_M  = 500
ZOOM_TIGHT_BUFFER  = 40
PRELOAD_AHEAD      = 10

# ── Colours (BGR) ────────────────────────────────────────────────────────────
C_BG        = (26,  13,  13)
C_CELL      = (38,  22,  22)
C_HEADER    = (42,  20,  10)
C_DIVIDER   = (0,  140, 220)
C_ICE_HIGH  = (70,  70, 255)
C_ICE_LOW   = (220, 170, 60)
C_WHITE     = (255, 255, 255)
C_DIM       = (160, 160, 190)
C_CONFIRMED = (60, 200, 80)


# ══════════════════════════════════════════════════════════════════════════════
#  UTILITIES
# ══════════════════════════════════════════════════════════════════════════════

def _norm_id(v: object) -> str:
    s = str(v)
    return s[:-2] if s.endswith(".0") else s


def _stretch(band: np.ndarray, lo: int = 2, hi: int = 98) -> np.ndarray:
    band  = band.astype(np.float32)
    valid = band > 0
    if not valid.any():
        return np.zeros_like(band, dtype=np.uint8)
    p_lo, p_hi = np.percentile(band[valid], (lo, hi))
    if p_hi <= p_lo:
        return np.zeros_like(band, dtype=np.uint8)
    out = np.clip((band - p_lo) / (p_hi - p_lo) * 255.0, 0, 255)
    out[~valid] = 0
    return out.astype(np.uint8)


def _put_text(img, text, x, y, scale=0.45, colour=C_WHITE, thickness=1):
    cv2.putText(img, text, (x, y), cv2.FONT_HERSHEY_SIMPLEX,
                scale, colour, thickness, cv2.LINE_AA)


def letterbox_frame(frame: np.ndarray, win_w: int, win_h: int) -> np.ndarray:
    fh, fw = frame.shape[:2]
    if fw == 0 or fh == 0 or win_w == 0 or win_h == 0:
        return frame
    scale   = min(win_w / fw, win_h / fh)
    new_w   = int(fw * scale)
    new_h   = int(fh * scale)
    interp  = cv2.INTER_AREA if scale < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(frame, (new_w, new_h), interpolation=interp)
    canvas  = np.zeros((win_h, win_w, 3), dtype=np.uint8)
    x0      = (win_w - new_w) // 2
    y0      = (win_h - new_h) // 2
    canvas[y0:y0 + new_h, x0:x0 + new_w] = resized
    return canvas


# ══════════════════════════════════════════════════════════════════════════════
#  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_very_cloudy(cloud_csvs=CLOUD_CSVS) -> set:
    names: set = set()
    for path in cloud_csvs:
        if not os.path.exists(path):
            print(f"  WARNING: cloud CSV not found: {path}")
            continue
        df   = pd.read_csv(path)
        mask = df["Classification"].str.strip() == "Very Cloudy"
        names.update(df.loc[mask, "Filename"].tolist())
    print(f"  {len(names):,} Very Cloudy filenames loaded.")
    return names


def load_timeseries(timeseries_csvs, very_cloudy: set) -> pd.DataFrame:
    frames = []
    for ds, path in timeseries_csvs.items():
        if not os.path.exists(path):
            print(f"  WARNING: timeseries CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = ds
        df["lake_id"] = df["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
        frames.append(df)
        print(f"  {ds}: {len(df):,} rows")
    if not frames:
        raise RuntimeError("No timeseries CSVs loaded.")
    out = pd.concat(frames, ignore_index=True)
    out["datetime"] = pd.to_datetime(out["datetime"])
    out = out[~out["filename"].isin(very_cloudy)].reset_index(drop=True)
    print(f"  -> {len(out):,} clean observations after removing cloudy images.")
    return out


def load_results(results_csvs) -> pd.DataFrame:
    frames = []
    for ds, path in results_csvs.items():
        if not os.path.exists(path):
            print(f"  WARNING: results CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = ds
        frames.append(df)
    if not frames:
        raise RuntimeError("No freeze-up results CSVs loaded.")
    out = pd.concat(frames, ignore_index=True)
    out["lake_id"] = out["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["year"]    = out["year"].astype(int)
    for col in ("freezeup_date", "obs_before_date", "obs_after_date"):
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce")
    # Keep only rows with a valid automated freeze-up date
    out = out[out["freezeup_date"].notna()].reset_index(drop=True)
    print(f"  -> {len(out):,} valid freeze-up results.")
    return out


def build_image_lookup(site_folder: str) -> dict:
    """
    Recursively index every .tif under site_folder.
    Works whether images are flat in the folder or nested inside
    year subfolders like Freezeup_2019/, Freezeup_2020/, etc.
    """
    lookup = {}
    for p in glob.glob(os.path.join(site_folder, "**", "*.tif"), recursive=True):
        lookup.setdefault(os.path.basename(p), p)
    return lookup


# ══════════════════════════════════════════════════════════════════════════════
#  LAKE-YEAR LIST BUILDER
# ══════════════════════════════════════════════════════════════════════════════

def build_lake_year_list(df_results: pd.DataFrame) -> list:
    rows = (df_results
            .sort_values(["dataset", "lake_id", "year"])
            .reset_index(drop=True))
    return [
        dict(lake_id=_norm_id(r["lake_id"]),
             year=int(r["year"]),
             dataset=str(r["dataset"]))
        for _, r in rows.iterrows()
    ]


# ══════════════════════════════════════════════════════════════════════════════
#  THUMBNAIL LOADER
# ══════════════════════════════════════════════════════════════════════════════

def load_thumbnail(tif_path: str,
                   lake_geom,
                   lake_crs,
                   thumb_w: int,
                   thumb_h: int,
                   buffer_m: float = BOUNDARY_BUFFER_M):
    try:
        with rasterio.open(tif_path) as src:
            lake_proj = (gpd.GeoSeries([lake_geom], crs=lake_crs)
                         .to_crs(src.crs).geometry.iloc[0])
            out_img, out_transform = rio_mask(src, [lake_proj.buffer(buffer_m)],
                                              crop=True, nodata=0)
            if out_img.shape[0] < 3:
                return None, []
            clip_h = out_img.shape[1]
            clip_w = out_img.shape[2]
            rgb = np.dstack([_stretch(out_img[2]),
                             _stretch(out_img[1]),
                             _stretch(out_img[0])])

        bgr   = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)
        thumb = cv2.resize(bgr, (thumb_w, thumb_h), interpolation=cv2.INTER_AREA)

        inv_t   = ~out_transform
        scale_x = thumb_w / clip_w
        scale_y = thumb_h / clip_h

        polys = (lake_proj.geoms
                 if lake_proj.geom_type == "MultiPolygon"
                 else [lake_proj])
        rings = []
        for poly in polys:
            for ring in [poly.exterior, *poly.interiors]:
                xs = np.array([c[0] for c in ring.coords])
                ys = np.array([c[1] for c in ring.coords])
                cols = inv_t.a * xs + inv_t.b * ys + inv_t.c
                rows = inv_t.d * xs + inv_t.e * ys + inv_t.f
                pts  = np.column_stack(
                    (np.round(cols * scale_x).astype(np.int32),
                     np.round(rows * scale_y).astype(np.int32))
                )
                rings.append(pts)

        return thumb, rings

    except Exception as exc:
        print(f"    Thumbnail error ({os.path.basename(tif_path)}): {exc}")
        return None, []


# ══════════════════════════════════════════════════════════════════════════════
#  LAKE-YEAR DATA CONTAINER
# ══════════════════════════════════════════════════════════════════════════════

class LakeYearData:
    __slots__ = ("lake_id", "year", "dataset",
                 "obs", "thumbs", "outlines",
                 "lake_geom", "lake_crs", "img_lookup",
                 "fu", "divider_pos",
                 "thumb_w", "thumb_h")

    def __init__(self, lake_id, year, dataset,
                 obs, thumbs, outlines, lake_geom, lake_crs, img_lookup,
                 fu, divider_pos, thumb_w, thumb_h):
        self.lake_id     = lake_id
        self.year        = year
        self.dataset     = dataset
        self.obs         = obs
        self.thumbs      = thumbs
        self.outlines    = outlines
        self.lake_geom   = lake_geom
        self.lake_crs    = lake_crs
        self.img_lookup  = img_lookup
        self.fu          = fu
        self.divider_pos = divider_pos
        self.thumb_w     = thumb_w
        self.thumb_h     = thumb_h

    @property
    def n(self):
        return len(self.obs)

    def move_left(self):
        self.divider_pos = max(0, self.divider_pos - 1)

    def move_right(self):
        self.divider_pos = min(self.n - 1, self.divider_pos + 1)


# ══════════════════════════════════════════════════════════════════════════════
#  PREPARATION  (called in background thread)
# ══════════════════════════════════════════════════════════════════════════════

def prepare_lake_year(task: dict,
                      df_clean: pd.DataFrame,
                      df_results: pd.DataFrame,
                      img_lookups: dict,
                      shp_cache: dict):
    lake_id = task["lake_id"]
    year    = task["year"]
    dataset = task["dataset"]

    obs_df = (df_clean[(df_clean["lake_id"] == lake_id) &
                       (df_clean["year"]    == year)    &
                       (df_clean["dataset"] == dataset)]
              .copy()
              .sort_values("datetime")
              .reset_index(drop=True))
    if obs_df.empty:
        return None

    fu_rows = df_results[(df_results["lake_id"] == lake_id) &
                         (df_results["year"]    == year)    &
                         (df_results["dataset"] == dataset)]
    if fu_rows.empty:
        return None
    fu = fu_rows.iloc[0].to_dict()

    shp_path = SITE_SHAPEFILES.get(dataset)
    if not shp_path:
        return None
    if not os.path.exists(shp_path):
        raise FileNotFoundError(
            f"Shapefile not found: {shp_path}\n"
            f"  Check that the path and folder name are correct under:\n"
            f"  {os.path.dirname(shp_path)}"
        )
    if shp_path not in shp_cache:
        shp_cache[shp_path] = gpd.read_file(shp_path)
    gdf = shp_cache[shp_path]
    id_col = next((c for c in ("id", "ID", "lake_id", "LAKE_ID", "FID", "OBJECTID")
                   if c in gdf.columns), None)
    if id_col is None:
        return None
    lake_rows = gdf[gdf[id_col].apply(_norm_id) == lake_id]
    if lake_rows.empty:
        return None
    lake_geom = lake_rows.geometry.iloc[0]
    lake_crs  = gdf.crs

    n           = len(obs_df)
    available_w = MAX_WINDOW_W - DIVIDER_W
    cell_w      = max(MIN_THUMB_W, available_w // n)
    thumb_w     = cell_w
    thumb_h     = max(80, int(280 * cell_w / 280))
    cell_h      = thumb_h + LABEL_H
    total_h     = HEADER_H + cell_h
    if total_h > MAX_WINDOW_H:
        scale   = (MAX_WINDOW_H - HEADER_H - LABEL_H) / thumb_h
        thumb_h = max(80, int(thumb_h * scale))
        thumb_w = max(MIN_THUMB_W, int(thumb_w * scale))

    img_lookup = img_lookups.get(dataset, {})
    obs_list   = obs_df.to_dict("records")
    thumbs     = []
    outlines   = []
    for row in obs_list:
        fname    = str(row.get("filename", ""))
        tif_path = img_lookup.get(fname)
        if tif_path:
            t, r = load_thumbnail(tif_path, lake_geom, lake_crs, thumb_w, thumb_h)
        else:
            t, r = None, []
        thumbs.append(t)
        outlines.append(r)

    obs_before_date = fu.get("obs_before_date")
    divider_pos = 0
    if obs_before_date is not None and not pd.isnull(obs_before_date):
        ref = pd.Timestamp(obs_before_date).normalize()
        for i, row in enumerate(obs_list):
            if pd.Timestamp(row["datetime"]).normalize() <= ref:
                divider_pos = i
    divider_pos = max(0, min(divider_pos, n - 1))

    return LakeYearData(lake_id=lake_id, year=year, dataset=dataset,
                        obs=obs_list, thumbs=thumbs, outlines=outlines,
                        lake_geom=lake_geom, lake_crs=lake_crs,
                        img_lookup=img_lookup, fu=fu, divider_pos=divider_pos,
                        thumb_w=thumb_w, thumb_h=thumb_h)


# ══════════════════════════════════════════════════════════════════════════════
#  BACKGROUND PRE-LOADER
# ══════════════════════════════════════════════════════════════════════════════

def preload_worker(task_q: queue.Queue, result_q: queue.Queue,
                   df_clean, df_results, img_lookups, shp_cache):
    while True:
        task = task_q.get()
        if task is None:
            break
        key  = (_norm_id(task["lake_id"]), task["year"], task["dataset"])
        data = None
        try:
            data = prepare_lake_year(task, df_clean, df_results,
                                     img_lookups, shp_cache)
        except Exception as exc:
            print(f"  Pre-load error {key}: {exc}")
        result_q.put((key, data))
        task_q.task_done()


# ══════════════════════════════════════════════════════════════════════════════
#  ZOOM RELOAD
# ══════════════════════════════════════════════════════════════════════════════

def _reload_thumbs(lyd: LakeYearData, buffer_m: float) -> None:
    new_thumbs   = []
    new_outlines = []
    for row in lyd.obs:
        fname    = str(row.get("filename", ""))
        tif_path = lyd.img_lookup.get(fname)
        if tif_path:
            t, r = load_thumbnail(tif_path, lyd.lake_geom, lyd.lake_crs,
                                  lyd.thumb_w, lyd.thumb_h, buffer_m)
        else:
            t, r = None, []
        new_thumbs.append(t)
        new_outlines.append(r)
    lyd.thumbs   = new_thumbs
    lyd.outlines = new_outlines


# ══════════════════════════════════════════════════════════════════════════════
#  RENDERING
# ══════════════════════════════════════════════════════════════════════════════

def _mid_date(lyd: LakeYearData):
    dp = lyd.divider_pos
    n  = lyd.n
    b  = lyd.obs[dp]         if 0 <= dp < n     else None
    a  = lyd.obs[dp + 1]     if 0 <= dp + 1 < n else None
    if b is None and a is None:
        return None
    if b is None:
        return pd.Timestamp(a["datetime"])
    if a is None:
        return pd.Timestamp(b["datetime"])
    t0, t1 = pd.Timestamp(b["datetime"]), pd.Timestamp(a["datetime"])
    return t0 + (t1 - t0) / 2


def _flanking_fnames(lyd: LakeYearData):
    dp = lyd.divider_pos
    n  = lyd.n
    bf = str(lyd.obs[dp]["filename"])       if 0 <= dp < n     else ""
    af = str(lyd.obs[dp + 1]["filename"])   if 0 <= dp + 1 < n else ""
    return bf, af


def render_frame(lyd: LakeYearData, current_idx: int, total: int,
                 show_outline: bool = True, zoom_tight: bool = False) -> np.ndarray:
    thumb_w = lyd.thumb_w
    thumb_h = lyd.thumb_h
    cell_h  = thumb_h + LABEL_H
    n       = lyd.n
    dp      = lyd.divider_pos

    canvas_w = n * thumb_w + DIVIDER_W
    canvas_h = HEADER_H + cell_h + 4

    frame = np.full((canvas_h, canvas_w, 3), C_BG, dtype=np.uint8)

    frame[:HEADER_H, :] = C_HEADER
    orig_dt = lyd.fu.get("freezeup_date")
    orig_s  = pd.Timestamp(orig_dt).strftime("%b %d %Y") if pd.notna(orig_dt) else "N/A"
    mid     = _mid_date(lyd)
    man_s   = mid.strftime("%b %d %Y") if mid is not None else "?"

    line1 = (f"  Lake {lyd.lake_id}   {lyd.year}   {lyd.dataset}     "
             f"Original freeze-up: {orig_s}     "
             f"Manual: {man_s}     "
             f"({current_idx} / {total})")
    zoom_s    = "ON (40m)" if zoom_tight   else "off"
    outline_s = "ON"      if show_outline else "off"
    line2 = (f"  Controls:  <- / A  left    -> / D  right    "
             f"Enter = save    X = no date    B = back    "
             f"O = outline [{outline_s}]    Z = zoom [{zoom_s}]    ESC = quit")
    _put_text(frame, line1, 6, 18, scale=0.50, colour=C_WHITE)
    _put_text(frame, line2, 6, 40, scale=0.40, colour=C_DIM)

    x  = 0
    cy = HEADER_H

    for i in range(n):
        if i == dp + 1:
            _draw_divider(frame, x, cy, DIVIDER_W, cell_h, lyd, man_s)
            x += DIVIDER_W

        frame[cy:cy + cell_h, x:x + thumb_w] = C_CELL

        thumb = lyd.thumbs[i]
        if thumb is not None:
            if show_outline and lyd.outlines[i]:
                thumb = thumb.copy()
                for pts in lyd.outlines[i]:
                    cv2.polylines(thumb, [pts], isClosed=True,
                                  color=(0, 0, 0), thickness=3, lineType=cv2.LINE_AA)
                for pts in lyd.outlines[i]:
                    cv2.polylines(thumb, [pts], isClosed=True,
                                  color=(0, 255, 255), thickness=1, lineType=cv2.LINE_AA)
            frame[cy:cy + thumb_h, x:x + thumb_w] = thumb
        else:
            _put_text(frame, "NOT FOUND",
                      x + thumb_w // 2 - 35, cy + thumb_h // 2,
                      scale=0.45, colour=(100, 100, 160))

        row    = lyd.obs[i]
        dt_str = pd.Timestamp(row["datetime"]).strftime("%b %d %Y")
        ly     = cy + thumb_h
        _put_text(frame, dt_str, x + 3, ly + 16, scale=0.44, colour=C_WHITE)

        ice = row.get("ice_fraction")
        if ice is not None:
            col = C_ICE_HIGH if ice >= FREEZE_THRESHOLD else C_ICE_LOW
            _put_text(frame, f"{ice:.1f}% ice",
                      x + 3, ly + 36, scale=0.44, colour=col)

        x += thumb_w

    if dp == n - 1:
        _draw_divider(frame, x, cy, DIVIDER_W, cell_h, lyd, man_s)

    return frame


def _draw_divider(frame, x, y, w, h, lyd, man_s):
    cv2.rectangle(frame, (x, y), (x + w - 1, y + h - 1), C_DIVIDER, -1)
    tx = x + w // 2 - 7
    for j, ch in enumerate("FREEZE"):
        _put_text(frame, ch, tx, y + 22 + j * 18, scale=0.40, colour=C_WHITE, thickness=1)
    _put_text(frame, "UP", tx - 2, y + 22 + 6 * 18, scale=0.40, colour=C_WHITE)
    _put_text(frame, man_s[:6], tx - 8, y + h - 28, scale=0.35, colour=(220, 220, 255))
    _put_text(frame, man_s[7:] if len(man_s) > 6 else "", tx - 5, y + h - 14,
              scale=0.35, colour=(220, 220, 255))


# ══════════════════════════════════════════════════════════════════════════════
#  OUTPUT CSV
# ══════════════════════════════════════════════════════════════════════════════

OUTPUT_COLS = ["study_site", "lake_id", "year",
               "original_freezeup_date", "manual_freezeup_date",
               "image_before", "image_after"]


def load_done(output_csv: str):
    if not os.path.exists(output_csv):
        return set(), []
    df   = pd.read_csv(output_csv)
    done = {(_norm_id(str(r["lake_id"])), int(r["year"]), str(r["study_site"]))
            for _, r in df.iterrows()}
    return done, df.to_dict("records")


def save_row(output_csv: str, record: dict, existing_rows: list):
    existing_rows.append(record)
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
        writer.writeheader()
        writer.writerows(existing_rows)


# ══════════════════════════════════════════════════════════════════════════════
#  KEY CODES
# ══════════════════════════════════════════════════════════════════════════════

_KEY_LEFT  = {2424832, 65361, 63234, ord('a'), ord('A')}   # 63234 = Mac left arrow
_KEY_RIGHT = {2555904, 65363, 63235, ord('d'), ord('D')}   # 63235 = Mac right arrow
_KEY_ENTER = {13, 10}
_KEY_ESC   = {27}
_KEY_X     = {ord('x'), ord('X')}
_KEY_B     = {ord('b'), ord('B')}
_KEY_O     = {ord('o'), ord('O')}
_KEY_Z     = {ord('z'), ord('Z')}


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run():
    print("=" * 60)

    # ── Sanity-check paths before doing any heavy work ───────────────────────
    print("\nChecking configured paths ...")
    ok = True
    for ds, p in SITE_SHAPEFILES.items():
        if os.path.exists(p):
            print(f"  [OK]  {ds} shapefile: {p}")
        else:
            print(f"  [MISSING]  {ds} shapefile: {p}")
            ok = False
    for ds, p in SITE_FOLDERS.items():
        if os.path.isdir(p):
            print(f"  [OK]  {ds} imagery folder: {p}")
        else:
            print(f"  [MISSING]  {ds} imagery folder: {p}")
            ok = False
    for ds, p in TIMESERIES_CSVS.items():
        if os.path.exists(p):
            print(f"  [OK]  {ds} timeseries CSV: {p}")
        else:
            print(f"  [MISSING]  {ds} timeseries CSV: {p}")
    for ds, p in FREEZEUP_RESULTS_CSVS.items():
        if os.path.exists(p):
            print(f"  [OK]  {ds} results CSV: {p}")
        else:
            print(f"  [MISSING]  {ds} results CSV: {p}")
    if not ok:
        print("\n  One or more required paths are missing.")
        print("  Please update YOUR_DRIVE_NAME and folder names in the USER CONFIG section.")
        return

    print("\nLoading cloud filter ...")
    very_cloudy = load_very_cloudy()

    print("\nLoading timeseries CSVs ...")
    df_clean = load_timeseries(TIMESERIES_CSVS, very_cloudy)

    print("\nLoading freeze-up results ...")
    df_results = load_results(FREEZEUP_RESULTS_CSVS)

    if "year" not in df_clean.columns:
        df_clean["year"] = df_clean["datetime"].dt.year

    print("\nBuilding lake-year list ...")
    all_lys = build_lake_year_list(df_results)
    if not all_lys:
        print("ERROR: No lake-years found in results CSV.")
        return

    done_set, existing_rows = load_done(OUTPUT_CSV)
    todo = [t for t in all_lys
            if (_norm_id(t["lake_id"]), t["year"], t["dataset"]) not in done_set]

    print(f"  Total: {len(all_lys)}   Already done: {len(done_set)}   "
          f"To review: {len(todo)}")
    if not todo:
        print("All lake-years already reviewed!")
        return

    print("\nIndexing TIF files ...")
    img_lookups = {}
    for ds in {t["dataset"] for t in todo}:
        folder = SITE_FOLDERS.get(ds)
        if folder and os.path.isdir(folder):
            img_lookups[ds] = build_image_lookup(folder)
            print(f"  {ds}: {len(img_lookups[ds]):,} TIFs indexed")
        else:
            print(f"  WARNING: imagery folder not found for {ds}: {folder}")

    N_WORKERS = 4
    shp_cache = {}
    task_q    = queue.Queue()
    result_q  = queue.Queue()

    for _w in range(N_WORKERS):
        t = threading.Thread(
            target=preload_worker,
            args=(task_q, result_q, df_clean, df_results, img_lookups, shp_cache),
            daemon=True,
            name=f"PreloadWorker-{_w}",
        )
        t.start()

    n_enqueued = 0
    for i in range(min(PRELOAD_AHEAD, len(todo))):
        task_q.put(todo[i])
        n_enqueued += 1

    preload_cache: dict = {}

    def _drain_results():
        while True:
            try:
                key, data = result_q.get_nowait()
                preload_cache[key] = data
            except queue.Empty:
                break

    def _get_lyd(idx: int):
        t   = todo[idx]
        key = (_norm_id(t["lake_id"]), t["year"], t["dataset"])
        while key not in preload_cache:
            _drain_results()
            if key not in preload_cache:
                try:
                    k2, data = result_q.get(timeout=0.15)
                    preload_cache[k2] = data
                except queue.Empty:
                    pass
        return preload_cache[key]

    WIN = "Freeze-up Manual Review"
    cv2.namedWindow(WIN, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(WIN, min(MAX_WINDOW_W, 1600), MAX_WINDOW_H)

    total = len(todo)
    idx   = 0
    shown = 0

    print("\n" + "=" * 60)
    print("OpenCV window opened. Use <- -> to move the divider, "
          "Enter to confirm, X for no freeze-up, ESC to quit.")
    print("=" * 60)

    while idx < total:
        t = todo[idx]

        next_to_enqueue = idx + PRELOAD_AHEAD
        while n_enqueued <= next_to_enqueue and n_enqueued < total:
            task_q.put(todo[n_enqueued])
            n_enqueued += 1

        print(f"  Loading [{idx+1}/{total}]: "
              f"Lake {t['lake_id']}  {t['year']}  {t['dataset']} ...",
              end="  ", flush=True)
        lyd = _get_lyd(idx)

        if lyd is None:
            print("SKIPPED (no data)")
            idx += 1
            continue
        print(f"{lyd.n} observations ready.")

        show_outline = True
        zoom_tight   = False
        # Seed with a safe size; only updated when getWindowImageRect returns
        # something plausible.  On Mac it can return 0 before the window has
        # fully composited, which causes the frame to render tiny then upscale,
        # producing severe pixelation.
        win_w, win_h = min(MAX_WINDOW_W, 1600), MAX_WINDOW_H

        while True:
            frame = render_frame(lyd, idx + 1, total,
                                 show_outline=show_outline, zoom_tight=zoom_tight)

            rect = cv2.getWindowImageRect(WIN)
            if rect[2] > 400 and rect[3] > 400:
                win_w, win_h = rect[2], rect[3]

            display = letterbox_frame(frame, win_w, win_h)

            cv2.imshow(WIN, display)
            key = cv2.waitKeyEx(30)

            if key == -1:
                continue

            if key in _KEY_LEFT:
                lyd.move_left()

            elif key in _KEY_RIGHT:
                lyd.move_right()

            elif key in _KEY_ENTER:
                mid   = _mid_date(lyd)
                man_s = mid.strftime("%Y-%m-%d") if mid is not None else ""
                bf, af = _flanking_fnames(lyd)
                orig   = lyd.fu.get("freezeup_date")
                orig_s = pd.Timestamp(orig).strftime("%Y-%m-%d") if pd.notna(orig) else ""
                record = {
                    "study_site":             lyd.dataset,
                    "lake_id":                lyd.lake_id,
                    "year":                   lyd.year,
                    "original_freezeup_date": orig_s,
                    "manual_freezeup_date":   man_s,
                    "image_before":           bf,
                    "image_after":            af,
                }
                save_row(OUTPUT_CSV, record, existing_rows)
                print(f"  + Saved  {lyd.lake_id}/{lyd.year}  ->  {man_s}")
                shown += 1
                idx   += 1
                break

            elif key in _KEY_X:
                orig   = lyd.fu.get("freezeup_date")
                orig_s = pd.Timestamp(orig).strftime("%Y-%m-%d") if pd.notna(orig) else ""
                record = {
                    "study_site":             lyd.dataset,
                    "lake_id":                lyd.lake_id,
                    "year":                   lyd.year,
                    "original_freezeup_date": orig_s,
                    "manual_freezeup_date":   "NO MANUAL DATE",
                    "image_before":           "",
                    "image_after":            "",
                }
                save_row(OUTPUT_CSV, record, existing_rows)
                print(f"  x Saved  {lyd.lake_id}/{lyd.year}  ->  NO MANUAL DATE")
                shown += 1
                idx   += 1
                break

            elif key in _KEY_B:
                if shown == 0:
                    print("  Nothing to undo this session.")
                else:
                    existing_rows.pop()
                    if existing_rows:
                        with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
                            writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
                            writer.writeheader()
                            writer.writerows(existing_rows)
                    else:
                        try:
                            os.remove(OUTPUT_CSV)
                        except OSError:
                            pass
                    shown -= 1
                    idx   -= 1
                    prev_t = todo[idx]
                    print(f"  Undone -- back to Lake {prev_t['lake_id']}  "
                          f"{prev_t['year']}  {prev_t['dataset']}")
                    break

            elif key in _KEY_O:
                show_outline = not show_outline

            elif key in _KEY_Z:
                zoom_tight = not zoom_tight
                buf = ZOOM_TIGHT_BUFFER if zoom_tight else BOUNDARY_BUFFER_M
                print(f"  Zoom {'tight (40 m)' if zoom_tight else 'default (500 m)'} "
                      f"-- reloading thumbnails ...", end="  ", flush=True)
                _reload_thumbs(lyd, buf)
                print("done.")

            elif key in _KEY_ESC:
                print(f"\n  ESC -- quitting after {shown} reviewed.")
                cv2.destroyAllWindows()
                for _ in range(N_WORKERS):
                    task_q.put(None)
                return

    cv2.destroyAllWindows()
    for _ in range(N_WORKERS):
        task_q.put(None)
    print(f"\n{'='*60}")
    print(f"  Completed!  {shown} lake-years reviewed.")
    print(f"  Results saved to: {OUTPUT_CSV}")
    print("=" * 60)


if __name__ == "__main__":
    run()


Checking configured paths ...
  [MISSING]  NS_50x50 shapefile: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Shapefiles\NS Lakes from ALPOD Shapefile\NS_50x50km_lakes.shp
  [MISSING]  YF_50x50 shapefile: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Shapefiles\YF Lakes from ALPOD Shapefile\YF_50x50km_lakes.shp
  [MISSING]  YKD_50x50 shapefile: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Shapefiles\YKD Lakes from ALPOD Shapefile\YKD_50x50km_lakes.shp
  [MISSING]  NS_50x50 imagery folder: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Imagery\NS 50x50 km
  [MISSING]  YF_50x50 imagery folder: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Imagery\YF 50x50 km
  [MISSING]  YKD_50x50 imagery folder: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Imagery\YKD 50x50 km
  [MISSING]  NS_50x50 timeseries CSV: /Volumes/COOL_Lab/Planet_API/Data/Freezeup Time Series\Freezeup CSVs\NS_freezeup_timeseries.csv
  [MISSING]  YF_50x50 timeseries CSV: /Vo

: 

## Transfer to Annie drive

In [ ]:
"""
prepare_undergrad_transfer.py
─────────────────────────────
Randomly samples N=250 lake-years per study site (NS, YF, YKD) where a
freeze-up date was detected, then copies all associated PlanetScope TIF
images to the undergrad's hard drive.

OUTPUT on destination drive:
  F:\Planet_API\Data\Freezeup Time Series Data\
    ├── NS 50x50 km\           ← mirrored sub-folder trees (TIF images)
    ├── YF 50x50 km\
    ├── YKD 50x50 km\
    ├── Data\                  ← filtered CSVs + shapefiles
    └── transfer_manifest.csv  ← study_site | lake_id | year | … | copied_paths

NOTES
  • Only lake-years with a valid (non-null) automated freeze-up date are eligible.
  • Sampling is of explicit (lake_id, year) pairs — a lake can appear 0, 1, or
    multiple times depending on the draw.
  • Each unique TIF is copied only once even if shared across lake-years.
  • Set RANDOM_SEED for reproducibility.
"""

import os
import glob
import shutil
import random
import warnings
from pathlib import Path

import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore")


# ══════════════════════════════════════════════════════════════════════════════
#  CONFIG
# ══════════════════════════════════════════════════════════════════════════════

N_SAMPLES   = 250       # lake-years to sample *per* study site
RANDOM_SEED = 42        # set None for a different draw each run

_OUT  = r"E:\planetscope_lake_ice\Data\Output\Freezeup"
_BASE = r"E:\planetscope_lake_ice\Data\Input\Study Sites - Manual ALPOD Data"

DEST_ROOT = r"F:\Planet_API\Data\Freezeup Time Series Data"

TIMESERIES_CSVS = {
    "NS_50x50":  os.path.join(_OUT, "NS_freezeup_timeseries.csv"),
    "YF_50x50":  os.path.join(_OUT, "YF_freezeup_timeseries.csv"),
    "YKD_50x50": os.path.join(_OUT, "YKD_freezeup_timeseries.csv"),
}

FREEZEUP_RESULTS_CSVS = {
    "NS_50x50":  os.path.join(_OUT, "NS_freezeup_results.csv"),
    "YF_50x50":  os.path.join(_OUT, "YF_freezeup_results.csv"),
    "YKD_50x50": os.path.join(_OUT, "YKD_freezeup_results.csv"),
}

CLOUD_CSVS = [
    os.path.join(_BASE, "YKD_cloud_classifications.csv"),
    os.path.join(_BASE, "NS_cloud_classifications.csv"),
    os.path.join(_BASE, "YF_cloud_classifications.csv"),
]

SITE_FOLDERS = {
    "NS_50x50":  os.path.join(_BASE, "NS 50x50 km"),
    "YF_50x50":  os.path.join(_BASE, "YF 50x50 km"),
    "YKD_50x50": os.path.join(_BASE, "YKD 50x50 km"),
}

SITE_SHAPEFILES = {
    "NS_50x50":  os.path.join(_BASE, "NS 50x50 km", "NS Lakes from ALPOD Shapefile", "NS_50x50km_lakes.shp"),
    "YF_50x50":  os.path.join(_BASE, "YF 50x50 km", "YF Lakes from ALPOD Shapefile", "YF_50x50km_lakes.shp"),
    "YKD_50x50": os.path.join(_BASE, "YKD 50x50 km", "YKD Lakes from ALPOD Shapefile", "YKD_50x50km_lakes.shp"),
}

SITE_FOLDER_NAMES = {
    "NS_50x50":  "NS 50x50 km",
    "YF_50x50":  "YF 50x50 km",
    "YKD_50x50": "YKD 50x50 km",
}

OUTPUT_MANIFEST = os.path.join(DEST_ROOT, "transfer_manifest.csv")


# ══════════════════════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def norm_id(v) -> str:
    s = str(v)
    return s[:-2] if s.endswith(".0") else s


def load_lake_ids(shp_path: str) -> list:
    gdf = gpd.read_file(shp_path)
    for col in ("id", "ID", "lake_id", "LAKE_ID", "FID", "OBJECTID"):
        if col in gdf.columns:
            if col != "id":
                print(f"  WARNING: 'id' column not found in {shp_path}; using '{col}'")
            return [norm_id(v) for v in gdf[col].tolist()]
    raise KeyError(f"No recognisable ID column in {shp_path}")


def load_very_cloudy() -> set:
    names = set()
    for path in CLOUD_CSVS:
        if not os.path.exists(path):
            print(f"  WARNING: cloud CSV not found: {path}")
            continue
        df   = pd.read_csv(path)
        mask = df["Classification"].str.strip() == "Very Cloudy"
        names.update(df.loc[mask, "Filename"].tolist())
    print(f"  {len(names):,} Very Cloudy filenames loaded.")
    return names


def load_timeseries(very_cloudy: set) -> pd.DataFrame:
    frames = []
    for ds, path in TIMESERIES_CSVS.items():
        if not os.path.exists(path):
            print(f"  WARNING: timeseries CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = ds
        df["lake_id"] = df["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
        frames.append(df)
        print(f"  {ds}: {len(df):,} rows")
    out = pd.concat(frames, ignore_index=True)
    out["datetime"] = pd.to_datetime(out["datetime"])
    out = out[~out["filename"].isin(very_cloudy)].reset_index(drop=True)
    print(f"  -> {len(out):,} clean observations after removing cloudy images.")
    return out


def load_results() -> pd.DataFrame:
    frames = []
    for ds, path in FREEZEUP_RESULTS_CSVS.items():
        if not os.path.exists(path):
            print(f"  WARNING: results CSV not found: {path}")
            continue
        df = pd.read_csv(path)
        if "dataset" not in df.columns:
            df["dataset"] = ds
        frames.append(df)
    out = pd.concat(frames, ignore_index=True)
    out["lake_id"]       = out["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
    out["year"]          = out["year"].astype(int)
    out["freezeup_date"] = pd.to_datetime(out["freezeup_date"], errors="coerce")
    out = out[out["freezeup_date"].notna()].reset_index(drop=True)
    print(f"  -> {len(out):,} lake-years with a detected freeze-up date.")
    return out


def build_image_lookup(site_folder: str) -> dict:
    lookup = {}
    for p in glob.glob(os.path.join(site_folder, "**", "*.tif"), recursive=True):
        lookup.setdefault(os.path.basename(p), p)
    return lookup


# ══════════════════════════════════════════════════════════════════════════════
#  SAMPLING  — explicit (lake_id, year) pairs
# ══════════════════════════════════════════════════════════════════════════════

def sample_lake_years(df_results: pd.DataFrame,
                      lake_ids_by_site: dict,
                      n: int,
                      seed) -> pd.DataFrame:
    """
    For each study site restrict df_results to lakes that exist in the
    shapefile, then draw n (lake_id, year) pairs at random without replacement.
    """
    rng    = random.Random(seed)
    frames = []

    for site, lake_ids in lake_ids_by_site.items():
        eligible = df_results[
            (df_results["dataset"] == site) &
            (df_results["lake_id"].isin(set(lake_ids)))
        ].copy()

        if len(eligible) == 0:
            print(f"  WARNING: {site} — no eligible lake-years found.")
            continue

        if len(eligible) < n:
            print(f"  WARNING: {site} only has {len(eligible)} eligible lake-years "
                  f"(requested {n}). Taking all of them.")
            sampled = eligible
        else:
            chosen  = rng.sample(range(len(eligible)), n)
            sampled = eligible.iloc[sorted(chosen)]

        print(f"  {site}: {len(sampled)} lake-years sampled "
              f"(pool = {len(eligible)} eligible lake-years across "
              f"{eligible['lake_id'].nunique()} unique lakes)")
        frames.append(sampled)

    return pd.concat(frames, ignore_index=True)


# ══════════════════════════════════════════════════════════════════════════════
#  SPACE ESTIMATOR
# ══════════════════════════════════════════════════════════════════════════════

def estimate_transfer_size(sampled: pd.DataFrame,
                           df_timeseries: pd.DataFrame,
                           img_lookups: dict):
    """
    Walk the sampled lake-years, resolve every TIF filename to a source path,
    and sum file sizes — counting each unique source path only once.
    Returns (size_by_site, total_bytes, n_unique_files).
    """
    seen        = set()   # deduplicate shared TIFs
    size_by_site = {ds: 0 for ds in sampled["dataset"].unique()}
    total_bytes  = 0

    for _, srow in sampled.iterrows():
        lake_id = norm_id(srow["lake_id"])
        year    = int(srow["year"])
        dataset = str(srow["dataset"])

        obs = df_timeseries[
            (df_timeseries["lake_id"] == lake_id) &
            (df_timeseries["year"]    == year)    &
            (df_timeseries["dataset"] == dataset)
        ]
        lookup = img_lookups.get(dataset, {})
        for _, obs_row in obs.iterrows():
            src = lookup.get(str(obs_row.get("filename", "")))
            if src and src not in seen:
                seen.add(src)
                try:
                    sz = os.path.getsize(src)
                    size_by_site[dataset] += sz
                    total_bytes += sz
                except OSError:
                    pass

    return size_by_site, total_bytes, len(seen)


# ══════════════════════════════════════════════════════════════════════════════
#  COPY ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def copy_images_for_sample(sampled: pd.DataFrame,
                           df_timeseries: pd.DataFrame,
                           img_lookups: dict) -> pd.DataFrame:
    """
    For every (lake_id, year, dataset) in sampled find all observations,
    copy each unique TIF once to DEST_ROOT preserving sub-folder structure,
    and return a manifest DataFrame.
    """
    copied_files  = {}   # src_path -> dest_path  (dedup guard)
    manifest_rows = []
    total_ly  = len(sampled)
    copied_n  = 0
    skipped_n = 0

    for row_i, (_, srow) in enumerate(sampled.iterrows(), 1):
        lake_id = norm_id(srow["lake_id"])
        year    = int(srow["year"])
        dataset = str(srow["dataset"])

        obs = df_timeseries[
            (df_timeseries["lake_id"] == lake_id) &
            (df_timeseries["year"]    == year)    &
            (df_timeseries["dataset"] == dataset)
        ].sort_values("datetime")

        if obs.empty:
            print(f"  [{row_i}/{total_ly}] No observations: "
                  f"{dataset} / Lake {lake_id} / {year}")
            continue

        lookup           = img_lookups.get(dataset, {})
        site_folder_root = SITE_FOLDERS[dataset]
        site_folder_name = SITE_FOLDER_NAMES[dataset]
        obs_dest_paths   = []

        for _, obs_row in obs.iterrows():
            fname = str(obs_row.get("filename", ""))
            src   = lookup.get(fname)

            if src is None:
                print(f"    NOT FOUND: {fname}")
                obs_dest_paths.append("")
                continue

            if src in copied_files:
                dest = copied_files[src]
                skipped_n += 1
            else:
                try:
                    rel = os.path.relpath(src, site_folder_root)
                except ValueError:
                    rel = os.path.basename(src)   # different-drive edge case
                dest = os.path.join(DEST_ROOT, site_folder_name, rel)
                os.makedirs(os.path.dirname(dest), exist_ok=True)
                shutil.copy2(src, dest)
                copied_files[src] = dest
                copied_n += 1

            obs_dest_paths.append(dest)

        freezeup_date = srow.get("freezeup_date", "")
        if pd.notna(freezeup_date):
            freezeup_date = pd.Timestamp(freezeup_date).strftime("%Y-%m-%d")
        else:
            freezeup_date = ""

        manifest_rows.append({
            "study_site":     dataset,
            "lake_id":        lake_id,
            "year":           year,
            "freezeup_date":  freezeup_date,
            "n_observations": len(obs),
            "copied_paths":   "|".join(p for p in obs_dest_paths if p),
        })

        if row_i % 50 == 0 or row_i == total_ly:
            print(f"  Progress: {row_i}/{total_ly} lake-years  "
                  f"({copied_n} new TIFs copied, {skipped_n} deduped)")

    print(f"\n  Done -- {copied_n} unique TIFs copied, "
          f"{skipped_n} shared-observation dedupes.")
    return pd.DataFrame(manifest_rows)


# ══════════════════════════════════════════════════════════════════════════════
#  SUPPORT FILES  (filtered CSVs + shapefiles)
# ══════════════════════════════════════════════════════════════════════════════

def copy_support_files(sampled: pd.DataFrame) -> None:
    out_data = os.path.join(DEST_ROOT, "Data")
    os.makedirs(out_data, exist_ok=True)

    sampled_keys = set(
        zip(sampled["dataset"], sampled["lake_id"], sampled["year"].astype(int))
    )

    for ds, src_path in TIMESERIES_CSVS.items():
        if not os.path.exists(src_path):
            continue
        df = pd.read_csv(src_path)
        df["lake_id"] = df["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
        if "dataset" not in df.columns:
            df["dataset"] = ds
        df["_year"] = pd.to_datetime(df["datetime"]).dt.year
        mask = df.apply(
            lambda r: (r["dataset"], norm_id(r["lake_id"]), int(r["_year"])) in sampled_keys,
            axis=1
        )
        out_path = os.path.join(out_data, os.path.basename(src_path))
        df[mask].drop(columns=["_year"]).to_csv(out_path, index=False)
        print(f"  Timeseries CSV ({ds}): {mask.sum()} rows -> {out_path}")

    for ds, src_path in FREEZEUP_RESULTS_CSVS.items():
        if not os.path.exists(src_path):
            continue
        df = pd.read_csv(src_path)
        df["lake_id"] = df["lake_id"].astype(str).str.replace(r"\.0$", "", regex=True)
        if "dataset" not in df.columns:
            df["dataset"] = ds
        df["year"] = df["year"].astype(int)
        mask = df.apply(
            lambda r: (r["dataset"], norm_id(r["lake_id"]), int(r["year"])) in sampled_keys,
            axis=1
        )
        out_path = os.path.join(out_data, os.path.basename(src_path))
        df[mask].to_csv(out_path, index=False)
        print(f"  Results CSV    ({ds}): {mask.sum()} rows -> {out_path}")

    for src_path in CLOUD_CSVS:
        if not os.path.exists(src_path):
            continue
        dest = os.path.join(out_data, os.path.basename(src_path))
        shutil.copy2(src_path, dest)
        print(f"  Cloud CSV: {dest}")

    for ds, src_shp in SITE_SHAPEFILES.items():
        shp_dir  = os.path.dirname(src_shp)
        shp_stem = Path(src_shp).stem
        rel_dir  = os.path.relpath(shp_dir, _BASE)
        dest_dir = os.path.join(out_data, "Shapefiles", rel_dir)
        os.makedirs(dest_dir, exist_ok=True)
        for ext in (".shp", ".shx", ".dbf", ".prj", ".cpg", ".qpj"):
            src = os.path.join(shp_dir, shp_stem + ext)
            if os.path.exists(src):
                shutil.copy2(src, os.path.join(dest_dir, shp_stem + ext))
        print(f"  Shapefile ({ds}) -> {dest_dir}")


# ══════════════════════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run():
    print("=" * 65)
    print(f"  Freeze-up Transfer Prep  --  N={N_SAMPLES} per study site")
    print("=" * 65)

    print("\n[1/6] Reading ALPOD shapefiles for lake ID lists ...")
    lake_ids_by_site = {}
    for ds, shp_path in SITE_SHAPEFILES.items():
        if not os.path.exists(shp_path):
            print(f"  ERROR: shapefile not found: {shp_path}")
            return
        ids = load_lake_ids(shp_path)
        lake_ids_by_site[ds] = ids
        print(f"  {ds}: {len(ids):,} lakes")

    print("\n[2/6] Loading cloud filter ...")
    very_cloudy = load_very_cloudy()

    print("\n[3/6] Loading timeseries & freeze-up results ...")
    df_timeseries = load_timeseries(very_cloudy)
    df_results    = load_results()

    print(f"\n[4/6] Sampling {N_SAMPLES} lake-years per site (seed={RANDOM_SEED}) ...")
    sampled = sample_lake_years(df_results, lake_ids_by_site,
                                n=N_SAMPLES, seed=RANDOM_SEED)
    print(f"  Total sampled: {len(sampled)} lake-years")

    print("\n[5/6] Indexing source TIF files ...")
    img_lookups = {}
    for ds in sampled["dataset"].unique():
        folder = SITE_FOLDERS.get(ds)
        if folder and os.path.isdir(folder):
            img_lookups[ds] = build_image_lookup(folder)
            print(f"  {ds}: {len(img_lookups[ds]):,} TIFs indexed")
        else:
            print(f"  WARNING: folder not found for {ds}: {folder}")
            img_lookups[ds] = {}

    print(f"\n[6/7] Estimating transfer size ...")
    size_by_site, total_bytes, n_unique = estimate_transfer_size(
        sampled, df_timeseries, img_lookups
    )
    for ds, nbytes in size_by_site.items():
        print(f"  {ds:<12}  {nbytes / 1e9:.2f} GB")
    print(f"  {'TOTAL':<12}  {total_bytes / 1e9:.2f} GB  ({n_unique:,} unique TIFs)")

    try:
        free = shutil.disk_usage(os.path.splitdrive(DEST_ROOT)[0] or DEST_ROOT).free
        print(f"  Free on destination drive: {free / 1e9:.2f} GB")
        if total_bytes > free:
            print("  ERROR: not enough free space on destination drive. Aborting.")
            return
    except Exception:
        pass  # can't check free space on this platform/path — proceed anyway

    answer = input("\n  Proceed with copy? [y/N]: ").strip().lower()
    if answer != "y":
        print("  Aborted.")
        return

    print(f"\n[7/7] Copying images -> {DEST_ROOT} ...")
    os.makedirs(DEST_ROOT, exist_ok=True)
    manifest = copy_images_for_sample(sampled, df_timeseries, img_lookups)

    print("\n  Copying support files (CSVs + shapefiles) ...")
    copy_support_files(sampled)

    manifest.to_csv(OUTPUT_MANIFEST, index=False)
    print(f"\n  Manifest: {OUTPUT_MANIFEST}")

    print(f"\n{'='*65}")
    print("  TRANSFER COMPLETE")
    print(f"{'='*65}")
    for ds in manifest["study_site"].unique():
        sub     = manifest[manifest["study_site"] == ds]
        n_files = sum(len(p.split("|")) for p in sub["copied_paths"] if p)
        print(f"  {ds:<12}  {len(sub):>3} lake-years  "
              f"{sub['lake_id'].nunique():>3} unique lakes  "
              f"{n_files:>5} observation images")
    print(f"\n  Destination: {DEST_ROOT}")
    print(f"  Manifest:    {OUTPUT_MANIFEST}")
    print(f"{'='*65}")


if __name__ == "__main__":
    run()